# Build the Mansoura Mobility database model

This notebook contains the first-version PostgreSQL schema, the database connection, and an explicit project-table reset for clean rebuilds.

## Before running

1. Copy `.env.example` to `.env` and set the local PostgreSQL values.
2. Install this project and the MAna database extra as described in `README.md`.
3. Create the empty `mansoura_mobility` database.
4. Run cells from top to bottom because foreign keys depend on earlier tables.
5. The reset cell deletes the existing project tables. Disable it before keeping generated data.

In [51]:
import os
from pathlib import Path

from dotenv import load_dotenv

project_root = Path(r"C:\Python\projects\pgSQL")
env_path = project_root / ".env"
if not env_path.exists():
    raise FileNotFoundError(
        f"Missing {env_path}. Create it from .env.example and add the local PostgreSQL credentials."
    )

load_dotenv(env_path, override=True)
if not os.getenv("PGDATABASE"):
    raise ValueError("PGDATABASE is missing from .env.")
if os.getenv("PGPASSWORD") == "change_me":
    raise ValueError("PGPASSWORD still contains the example value.")

from MAna.database import connect_postgres, execute_query, list_postgres_tables

db = connect_postgres(driver="psycopg2")

[OK] Connected to database: postgresql+psycopg2://localhost/mansoura_mobility


In [52]:
connection_check = execute_query(
    """
    SELECT
        current_database() AS database_name,
        current_user AS user_name,
        current_setting('TimeZone') AS session_timezone,
        version() AS postgres_version;
    """,
    db,
    return_results=True,
)
connection_check

,database_name,user_name,session_timezone,postgres_version
0,mansoura_mobility,postgres,Africa/Cairo,"PostgreSQL 15.0, compiled by Visual C++ build ..."


## Reset the project tables

**Warning:** The reset statement is disabled. Enabling it deletes every Mansoura Mobility table and all data inside them. The statement names only this project's tables instead of dropping the entire `public` schema.

In [53]:
drop_project_tables_sql = """
DROP TABLE IF EXISTS
    reports,
    driver_passenger_ratings,
    passenger_driver_ratings,
    refunds,
    payment_attempts,
    rides,
    offers,
    zone_routes,
    vehicles,
    zones,
    drivers,
    passengers,
    accounts
CASCADE;
"""

# execute_query(drop_project_tables_sql, db)

[OK] Query executed successfully


## Schema conventions

Each table definition remains visible and runs in foreign-key dependency order.

The schema uses `SERIAL` identifiers, `TIMESTAMPTZ` for operational timestamps, `DECIMAL(p, s)` for measured values, and text columns with `CHECK` constraints for first-version statuses.

In [54]:
def run_ddl(sql: str) -> None:
    if "CREATE TABLE" not in sql.upper():
        raise ValueError("DDL must contain a CREATE TABLE statement.")
    execute_query(sql, db)

## 1. `accounts`

Required columns: `account_id`, `full_name`, `phone_number`, `email`, `signup_at`, `account_status`, `last_seen_at`.

Constraints: generated primary key, required name, unique phone and email, at least one contact method, status limited to `ACTIVE`, `INACTIVE`, `SUSPENDED`, and timestamp defaults. Email uniqueness is currently case-sensitive.

In [55]:
accounts_sql = """
CREATE TABLE accounts (
    account_id SERIAL PRIMARY KEY,
    full_name VARCHAR(100) NOT NULL,
    phone_number VARCHAR(20) UNIQUE,
    email VARCHAR(100) UNIQUE,
    signup_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    account_status VARCHAR(20) CHECK (account_status IN ('ACTIVE', 'INACTIVE', 'SUSPENDED')),
    last_seen_at TIMESTAMPTZ,
    CHECK (phone_number IS NOT NULL OR email IS NOT NULL)
);
"""

run_ddl(accounts_sql)

[OK] Query executed successfully


## 2. `passengers`

Required column: `passenger_id`.

This is a shared-primary-key role table: `passenger_id` is both its primary key and a foreign key to `accounts.account_id`. One account may appear here once.

In [56]:
passengers_sql = """
CREATE TABLE passengers (
    passenger_id INT PRIMARY KEY REFERENCES accounts(account_id) ON DELETE CASCADE
);
"""

run_ddl(passengers_sql)

[OK] Query executed successfully


## 3. `drivers`

Required columns: `driver_id`, `license_number`, `is_accepting_offers`.

`driver_id` is both primary key and foreign key to `accounts.account_id`. Give `is_accepting_offers` a sensible required default. An account may exist in both `passengers` and `drivers`.

In [57]:
drivers_sql = """
CREATE TABLE drivers (
    driver_id INT PRIMARY KEY REFERENCES accounts(account_id) ON DELETE CASCADE,
    license_number VARCHAR(50) UNIQUE NOT NULL,
    is_accepting_offers BOOLEAN DEFAULT TRUE);
"""

run_ddl(drivers_sql)

[OK] Query executed successfully


## 4. `zones`

Required columns: `zone_id`, `zone_name`, `city_name`, `zone_type`, `center_latitude`, `center_longitude`.

Rules: generated primary key; `(city_name, zone_name)` unique together; zone type limited to `RESIDENTIAL`, `COMMERCIAL`, `INDUSTRIAL`, `MIXED`, `TRANSPORT_HUB`; latitude and longitude within valid world ranges.

In [58]:
zones_sql = """
CREATE TABLE zones (
    zone_id SERIAL PRIMARY KEY,
    zone_name VARCHAR(100) NOT NULL,
    city_name VARCHAR(100) NOT NULL,
    UNIQUE (city_name, zone_name),
    zone_type VARCHAR(50) CHECK (zone_type IN ('RESIDENTIAL', 'COMMERCIAL', 'INDUSTRIAL', 'MIXED', 'TRANSPORT_HUB')),
    center_latitude DECIMAL(9, 6) CHECK (center_latitude BETWEEN -90 AND 90),
    center_longitude DECIMAL(9, 6) CHECK (center_longitude BETWEEN -180 AND 180));
"""

run_ddl(zones_sql)

[OK] Query executed successfully


## 5. `vehicles`

Required columns: `vehicle_id`, `driver_id`, `brand`, `model`, `model_year`, `vehicle_category`, `plate_number`, `service_status`, `registered_at`; optional `retired_at`.

Rules: generated primary key; driver foreign key; unique plate; categories `ECONOMY`, `COMFORT`, `PREMIUM`; statuses `ACTIVE`, `INACTIVE`, `OUT_OF_SERVICE`; plausible model year; retirement cannot precede registration.

In [59]:
vehicles_sql = """
CREATE TABLE vehicles (
    vehicle_id SERIAL PRIMARY KEY,
    driver_id INT REFERENCES drivers(driver_id) ON DELETE CASCADE,
    brand VARCHAR(50) NOT NULL,
    model VARCHAR(50) NOT NULL,
    model_year INT CHECK (model_year >= 1886 AND model_year <= EXTRACT(YEAR FROM CURRENT_DATE)),
    vehicle_category VARCHAR(50) CHECK (vehicle_category IN ('ECONOMY', 'COMFORT', 'PREMIUM')),
    plate_number VARCHAR(20) UNIQUE NOT NULL,
    service_status VARCHAR(20) CHECK (service_status IN ('ACTIVE', 'INACTIVE', 'OUT_OF_SERVICE')),
    registered_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    retired_at TIMESTAMPTZ,
    CHECK (retired_at IS NULL OR retired_at >= registered_at),
    UNIQUE (driver_id, vehicle_id));
"""

run_ddl(vehicles_sql)

[OK] Query executed successfully


## 6. `zone_routes`

Required columns: `origin_zone_id`, `destination_zone_id`, `baseline_distance_km`, `baseline_duration_minutes`.

Rules: both zone foreign keys; composite primary key; positive distance and duration. Same-zone routes are allowed, and A-to-B differs from B-to-A.

In [60]:
zone_routes_sql = """
CREATE TABLE zone_routes (
    origin_zone_id INT REFERENCES zones(zone_id) ON DELETE CASCADE,
    destination_zone_id INT REFERENCES zones(zone_id) ON DELETE CASCADE,
    baseline_distance_km DECIMAL(5, 2) CHECK (baseline_distance_km > 0),
    baseline_duration_minutes DECIMAL(5, 2) CHECK (baseline_duration_minutes > 0),
    PRIMARY KEY (origin_zone_id, destination_zone_id));
"""

run_ddl(zone_routes_sql)

[OK] Query executed successfully


## 7. `offers`

Required columns: `offer_id`, `passenger_id`, `driver_id`, `vehicle_id`, `pickup_zone_id`, `dropoff_zone_id`, `initial_fare_egp`, `offer_status`, `initiated_at`; optional `declined_by`, `decline_reason`, `decided_at`.

Constraints: foreign keys, positive fare, controlled statuses, empty decline fields after acceptance, a decision time for closed offers, and a composite driver-vehicle reference that proves vehicle ownership.

In [61]:
offers_sql = """
CREATE TABLE offers (
    offer_id SERIAL PRIMARY KEY,
    passenger_id INT REFERENCES passengers(passenger_id),
    driver_id INT,
    vehicle_id INT,
    pickup_zone_id INT REFERENCES zones(zone_id),
    dropoff_zone_id INT REFERENCES zones(zone_id),
    initial_fare_egp DECIMAL(10, 2) CHECK (initial_fare_egp > 25),
    offer_status VARCHAR(20) CHECK (offer_status IN ('PENDING', 'ACCEPTED', 'DECLINED', 'NEGOTIATING', 'WITHDRAWN', 'AUTO_CANCELLED')),
    initiated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    declined_by VARCHAR(20) CHECK (declined_by IN ('PASSENGER', 'DRIVER', 'SYSTEM')),
    decided_at TIMESTAMPTZ,
    decline_reason VARCHAR(255),
    FOREIGN KEY (driver_id, vehicle_id) REFERENCES vehicles(driver_id, vehicle_id),
    CHECK (pickup_zone_id <> dropoff_zone_id),
    CHECK (offer_status IN ('PENDING', 'NEGOTIATING') OR decided_at IS NOT NULL),
    CHECK (offer_status <> 'ACCEPTED' OR (declined_by IS NULL AND decline_reason IS NULL)),
    CHECK (offer_status NOT IN ('DECLINED', 'WITHDRAWN', 'AUTO_CANCELLED') OR (declined_by IS NOT NULL AND decline_reason IS NOT NULL))
);
"""

run_ddl(offers_sql)

[OK] Query executed successfully


## 8. `rides`

Required columns: `ride_id`, `offer_id`; current first-version fields also include `ride_status`, `started_at`, `start_distance_metres`, `completed_at`, `final_fare_egp`, `cancelled_by`, `cancellation_stage`, `cancellation_reason`.

Rules: generated primary key; one required unique offer per ride; statuses `AWAITING_PAYMENT`, `DRIVER_EN_ROUTE`, `READY_TO_START`, `IN_PROGRESS`, `COMPLETED`, `CANCELLED_BEFORE_START`, `TERMINATED_EARLY`; non-negative measurements. The fuller milestone design and cancellation consistency rules are intentionally deferred. A normal `CHECK` cannot inspect whether the referenced offer is `ACCEPTED`; record that as a later transaction/trigger rule.

In [62]:
rides_sql = """
CREATE TABLE rides (
    ride_id SERIAL PRIMARY KEY,
    offer_id INT NOT NULL UNIQUE REFERENCES offers(offer_id),
    final_fare_egp DECIMAL(10, 2) CHECK (final_fare_egp >= 0),
    ride_status VARCHAR(30) CHECK (ride_status IN ('AWAITING_PAYMENT', 'DRIVER_EN_ROUTE', 'READY_TO_START', 'IN_PROGRESS', 'COMPLETED', 'CANCELLED_BEFORE_START', 'TERMINATED_EARLY')),
    started_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    start_distance_metres DECIMAL(5, 2) CHECK (start_distance_metres >= 0),
    completed_at TIMESTAMP,
    cancelled_by VARCHAR(20) CHECK (cancelled_by IN ('PASSENGER', 'DRIVER', 'SYSTEM')),
    cancellation_reason VARCHAR(255),
    cancellation_stage VARCHAR(30) CHECK (cancellation_stage IN ('BEFORE_DRIVER_ARRIVAL', 'AFTER_DRIVER_ARRIVAL', 'DURING_RIDE'))
);
"""

run_ddl(rides_sql)

[OK] Query executed successfully


## 9. `payment_attempts`

Required columns: `payment_attempt_id`, `ride_id`, `attempt_number`, `payment_method`, `payment_status`, `requested_amount_egp`, `provider_reference`, `initiated_at`; optional authorization/capture amounts, failure reason, and milestone times.

Rules: ride foreign key; `(ride_id, attempt_number)` unique; provider reference unique; methods `CARD`, `WALLET`; statuses `INITIATED`, `AUTHORIZED`, `FAILED`, `CAPTURED`, `RELEASED`; positive amounts; failed attempts require a failure reason.

In [63]:
payment_attempts_sql = """
CREATE TABLE payment_attempts (
    payment_attempt_id SERIAL PRIMARY KEY,
    ride_id INT REFERENCES rides(ride_id),
    attempt_number INT CHECK (attempt_number > 0),
    UNIQUE (ride_id, attempt_number),
    payment_method VARCHAR(20) CHECK (payment_method IN ('CARD', 'WALLET')),
    payment_status VARCHAR(20) CHECK (payment_status IN ('INITIATED', 'AUTHORIZED', 'FAILED', 'CAPTURED', 'RELEASED')),
    requested_amount_egp DECIMAL(10, 2) CHECK (requested_amount_egp > 25),
    authorized_amount_egp DECIMAL(10, 2) CHECK (authorized_amount_egp > 25),
    captured_amount_egp DECIMAL(10, 2) CHECK (captured_amount_egp > 25),
    provider_reference VARCHAR(100) UNIQUE,
    failure_reason VARCHAR(255),
    initiated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    authorized_at TIMESTAMPTZ,
    captured_at TIMESTAMPTZ,
    released_at TIMESTAMPTZ,
    CHECK (payment_status <> 'FAILED' OR failure_reason IS NOT NULL));
"""

run_ddl(payment_attempts_sql)

[OK] Query executed successfully


## 10. `refunds`

Required columns: `refund_id`, `payment_attempt_id`, `refund_amount_egp`, `refund_reason`, `refund_status`, `requested_at`; optional `completed_at`, `failure_reason`.

Constraints: payment foreign key, positive refund amount, controlled statuses, and completion/failure fields consistent with status. Limiting total refunds to captured funds requires transaction-level enforcement.

In [64]:
refunds_sql = """
CREATE TABLE refunds (
    refund_id SERIAL PRIMARY KEY,
    payment_attempt_id INT REFERENCES payment_attempts(payment_attempt_id),
    refund_amount_egp DECIMAL(10, 2) CHECK (refund_amount_egp > 0),
    refund_reason VARCHAR(255),
    refund_status VARCHAR(20) CHECK (refund_status IN ('REQUESTED', 'PROCESSING', 'COMPLETED', 'FAILED')),
    requested_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    completed_at TIMESTAMPTZ,
    failure_reason VARCHAR(255),
    CHECK (refund_status <> 'COMPLETED' OR completed_at IS NOT NULL),
    CHECK (refund_status <> 'FAILED' OR failure_reason IS NOT NULL)
);
"""

run_ddl(refunds_sql)

[OK] Query executed successfully


## 11. `passenger_driver_ratings`

Columns: `ride_id`, `attitude_score`, `driving_safety_score`, `vehicle_cleanliness_score`, `comfort_score`, `route_quality_score`, `submitted_at`; optional `comment`.

Rules: `ride_id` is the primary key and foreign key; every supplied score is between 1 and 5. The table direction identifies who rates whom, so passenger and driver IDs are not duplicated here.

In [65]:
passenger_driver_ratings_sql = """
CREATE TABLE passenger_driver_ratings (
    ride_id INT REFERENCES rides(ride_id) PRIMARY KEY,
    attitude_score INT CHECK (attitude_score BETWEEN 1 AND 5),
    driving_safety_score INT CHECK (driving_safety_score BETWEEN 1 AND 5),
    vehicle_cleanliness_score INT CHECK (vehicle_cleanliness_score BETWEEN 1 AND 5),
    comfort_score INT CHECK (comfort_score BETWEEN 1 AND 5),
    route_quality_score INT CHECK (route_quality_score BETWEEN 1 AND 5),
    submitted_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    comment VARCHAR(255));

"""

run_ddl(passenger_driver_ratings_sql)

[OK] Query executed successfully


## 12. `driver_passenger_ratings`

Columns: `ride_id`, `attitude_score`, `punctuality_score`, `pickup_cooperation_score`, `respect_safety_score`, `submitted_at`; optional `comment`.

Rules: `ride_id` is the primary key and foreign key; every supplied score is between 1 and 5. The table direction identifies who rates whom.

In [66]:
driver_passenger_ratings_sql = """
CREATE TABLE driver_passenger_ratings (
    ride_id INT REFERENCES rides(ride_id) PRIMARY KEY,
    attitude_score INT CHECK (attitude_score BETWEEN 1 AND 5),
    punctuality_score INT CHECK (punctuality_score BETWEEN 1 AND 5),
    pickup_cooperation_score INT CHECK (pickup_cooperation_score BETWEEN 1 AND 5),
    respect_safety_score INT CHECK (respect_safety_score BETWEEN 1 AND 5),
    submitted_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    comment VARCHAR(255));
"""

run_ddl(driver_passenger_ratings_sql)

[OK] Query executed successfully


## 13. `reports`

Columns: `report_id`, `ride_id`, `opened_by`, `report_category`, `other_report_category`, `description`, `report_status`, `opened_at`, `resolved_at`, `resolution_notes`.

Rules: generated primary key; ride foreign key; opener `PASSENGER`, `DRIVER`, `SYSTEM`; allowed report categories and statuses from the logical model; resolution fields consistent with resolved/dismissed states.

In [67]:
reports_sql = """
CREATE TABLE reports (
    report_id SERIAL PRIMARY KEY,
    ride_id INT REFERENCES rides(ride_id),
    opened_by VARCHAR(20) CHECK (opened_by IN ('PASSENGER', 'DRIVER', 'SYSTEM')),
    report_category VARCHAR(50) CHECK (report_category IN ('PASSENGER_BEHAVIOR', 'DRIVER_BEHAVIOR', 'VEHICLE_ISSUE', 'PAYMENT_ISSUE', 'OTHER')),
    other_report_category VARCHAR(100),
    description TEXT,
    report_status VARCHAR(20) CHECK (report_status IN ('OPEN', 'IN_PROGRESS', 'RESOLVED', 'REJECTED')),
    opened_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP,
    resolved_at TIMESTAMPTZ,
    resolution_notes TEXT);
"""

run_ddl(reports_sql)

[OK] Query executed successfully


## Inspect the schema

The table inventory confirms that every schema statement completed. Constraint behavior is checked separately with invalid test inserts.

In [68]:
list_postgres_tables(db, schema="public")

,schema_name,table_name,estimated_rows,total_bytes,total_size
0,public,accounts,-1,24576,24 kB
1,public,payment_attempts,-1,24576,24 kB
2,public,vehicles,-1,24576,24 kB
3,public,drivers,-1,16384,16 kB
4,public,refunds,-1,16384,16 kB
5,public,reports,-1,16384,16 kB
6,public,rides,-1,16384,16 kB
7,public,zones,-1,16384,16 kB
8,public,driver_passenger_ratings,-1,8192,8192 bytes
9,public,offers,-1,8192,8192 bytes


## Validation targets

Validation covers duplicate phone/email, invalid account status, driver roles without accounts, vehicles without drivers, impossible coordinates, negative fares, reused offers, ratings outside 1–5, and negative refunds. Cross-table lifecycle rules remain part of the later transaction/trigger phase.